# Day 04: Data Handling — Datasets, Batching, Tokenization

**Goal:** Learn how text becomes numbers that a model can process.

### The gap we're bridging:

```
Days 1-3: input = [3.0, 8.0]          → model → score = 75.0
Day 4:    input = "The cat sat on the" → ???   → "mat"
```

Models only understand numbers. So before any text can enter a model, we need to:
1. **Tokenize** — split text into pieces and convert to numbers
2. **Batch** — group data into chunks that fit in memory
3. **Create input/target pairs** — the model predicts the next token

Let's build each step.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

## 1. The Fundamental Problem: Text → Numbers

A model does matrix math. It can process `[0.5, 0.8, 0.2]` but not `"hello"`.

So the FIRST step in any NLP pipeline is: **turn each piece of text into a number**.

### Character-level tokenization (simplest approach)

The idea: each unique character gets a number.

```
'a' → 0,  'b' → 1,  'c' → 2,  ...  'z' → 25
' ' → 26,  '.' → 27, ...
```

Then "cat" becomes `[2, 0, 19]`.

Let's build this from scratch:

In [ ]:
# Let's use a small text to learn on
text = """To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles."""

print(f"Text length: {len(text)} characters")
print(f"First 100 chars: '{text[:100]}...'\n")

# Step 1: Find all unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Unique characters ({vocab_size}): {chars}")
print(f"\nThis is our vocabulary — every character the model will know about.")

In [ ]:
# Step 2: Build the lookup tables
# We need TWO mappings:
#   char → number (for encoding: text → numbers)
#   number → char (for decoding: numbers → text)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print("Character → Number mapping (first 10):")
for ch, idx in list(char_to_idx.items())[:10]:
    display_ch = repr(ch)  # show special chars like \n
    print(f"  {display_ch:>6} → {idx}")

print(f"\nThis is called the 'vocabulary' or 'tokenizer'.")
print(f"GPT-2 has ~50,000 entries. Ours has {vocab_size}.")

In [ ]:
# Step 3: Encode and decode functions

def encode(text):
    """Convert a string to a list of token IDs."""
    return [char_to_idx[ch] for ch in text]

def decode(token_ids):
    """Convert a list of token IDs back to a string."""
    return ''.join([idx_to_char[i] for i in token_ids])

# Try it!
test = "To be"
encoded = encode(test)
decoded = decode(encoded)

print(f"Original text: '{test}'")
print(f"Encoded:        {encoded}")
print(f"Decoded back:  '{decoded}'")
print(f"Round-trip OK:  {test == decoded} ✓")

print(f"\nLet's trace each character:")
for ch, idx in zip(test, encoded):
    print(f"  '{ch}' → {idx}")

In [ ]:
# Step 4: Encode the ENTIRE text into a tensor of numbers
# This is what we'll feed to the model

data = torch.tensor(encode(text), dtype=torch.long)

print(f"Full text encoded to tensor:")
print(f"  Shape: {data.shape} ({len(data)} tokens)")
print(f"  dtype: {data.dtype} (long = integers, because these are IDs not decimals)")
print(f"  First 50 tokens: {data[:50].tolist()}")
print(f"  Decoded back:    '{decode(data[:50].tolist())}'")

print(f"\nThe ENTIRE Shakespeare passage is now just a list of {len(data)} numbers.")
print(f"This is what the model will see — never the raw text.")

## 2. Input/Target Pairs — What Does the Model Actually Learn?

A language model's job is: **given some tokens, predict the next token**.

So from the sequence `[T, o, _, b, e]`, we create:

```
When the model sees:  "T"        → it should predict: "o"
When the model sees:  "To"       → it should predict: " "
When the model sees:  "To "      → it should predict: "b"
When the model sees:  "To b"     → it should predict: "e"
```

In practice, we do this efficiently by shifting the sequence by 1:

```
Input:  [T, o, _, b, e]     (positions 0-4)
Target: [o, _, b, e, _]     (positions 1-5 — same data, shifted right by 1)
```

The target at each position is simply the **next character** in the text.

In [ ]:
# Let's see this concretely

block_size = 8  # how many characters the model sees at once (context window)

# Take a chunk of our data
chunk = data[:block_size + 1]  # +1 because we need one extra for the last target

print(f"Chunk of {block_size + 1} tokens: {chunk.tolist()}")
print(f"As text: '{decode(chunk.tolist())}'\n")

# The input is positions 0 to block_size-1
# The target is positions 1 to block_size (shifted by 1)
x = chunk[:block_size]    # input:  first 8 tokens
y = chunk[1:block_size+1] # target: next 8 tokens (shifted by 1)

print(f"Input (x):  {x.tolist()}")
print(f"Target (y): {y.tolist()}\n")

print("What the model learns from this one chunk:")
print(f"{'Position':>8} | {'Input so far':>20} | {'Should predict':>14}")
print("-" * 50)
for i in range(block_size):
    context = decode(x[:i+1].tolist())
    target_char = decode([y[i].item()])
    print(f"{i:>8} | {repr(context):>20} | {repr(target_char):>14}")

print(f"\n8 training examples from just 9 characters!")

### Why this is clever:

From just 9 characters, we got **8 training examples**. The model learns to predict the next character from contexts of length 1, 2, 3, ... up to 8. 

This means:
- It learns `"T" → "o"` (with 1 char of context)
- It learns `"To be or" → " "` (with 8 chars of context)
- Both from the same chunk of data

GPT-4's block size is ~128,000 tokens. Each chunk gives 128,000 training examples!

---

## 3. Batching — Processing Multiple Chunks at Once

We have one long text. We need to:
1. Cut it into chunks of `block_size`
2. Group chunks into batches
3. Feed one batch at a time to the model

This is more efficient because GPUs can process multiple chunks in parallel.

In [ ]:
# Let's build a batch of random chunks from our text

block_size = 8
batch_size = 4  # 4 chunks at a time

def get_batch(data, block_size, batch_size):
    """Pick random starting positions and create input/target pairs."""
    # Random starting indices (can't go past the end)
    max_start = len(data) - block_size - 1
    starts = torch.randint(0, max_start, (batch_size,))
    
    # For each start, grab block_size tokens (input) and the next block_size (target)
    x = torch.stack([data[s : s + block_size] for s in starts])
    y = torch.stack([data[s + 1 : s + block_size + 1] for s in starts])
    
    return x, y

# Get one batch
xb, yb = get_batch(data, block_size, batch_size)

print(f"Input batch shape:  {xb.shape}  (batch_size × block_size = {batch_size} × {block_size})")
print(f"Target batch shape: {yb.shape}  (same shape — one target per input position)")

print(f"\nEach row is one random chunk from the text:\n")
for i in range(batch_size):
    input_text = decode(xb[i].tolist())
    target_text = decode(yb[i].tolist())
    print(f"  Chunk {i}: input='{input_text}'  →  target='{target_text}'")

### Understanding the batch shape:

```
xb.shape = (4, 8)
            ↑  ↑
            |  └── block_size: 8 characters per chunk
            └── batch_size: 4 chunks processed in parallel

This means the GPU processes 4 × 8 = 32 next-character predictions in ONE forward pass!
```

In real training:
- batch_size = 32-512 (or more)
- block_size = 1024-128000
- So one forward pass might process **millions** of predictions at once

---

## 4. Train/Validation Split — Measuring Real Performance

We need to check if the model actually LEARNED the patterns, not just memorized the text. So we split data into:

- **Training set** (~90%) — the model trains on this
- **Validation set** (~10%) — we check performance on this (model never trains on it)

If loss is low on training but high on validation → the model just memorized (overfitting).
If loss is low on BOTH → the model actually learned the patterns.

In [ ]:
# Split data: 90% train, 10% validation
n = len(data)
split = int(0.9 * n)
train_data = data[:split]
val_data = data[split:]

print(f"Total tokens:      {n}")
print(f"Training tokens:   {len(train_data)} ({len(train_data)/n*100:.0f}%)")
print(f"Validation tokens: {len(val_data)} ({len(val_data)/n*100:.0f}%)")

print(f"\nTraining text:   '{decode(train_data[:60].tolist())}...'")
print(f"Validation text: '{decode(val_data[:60].tolist())}...'")

## 5. PyTorch Dataset & DataLoader — The Standard Way

What we built above works, but PyTorch provides `Dataset` and `DataLoader` classes that handle batching, shuffling, and parallel loading for you.

This is the pattern used in every real PyTorch project. Let's build a proper Dataset:

In [ ]:
# A PyTorch Dataset for character-level language modeling

class CharDataset(Dataset):
    """
    Takes a long tensor of token IDs and returns (input, target) chunks.
    
    For text = [10, 5, 3, 8, 2, 7, 1, 4, 9, 6] and block_size=4:
      Sample 0: input=[10,5,3,8]  target=[5,3,8,2]
      Sample 1: input=[5,3,8,2]   target=[3,8,2,7]
      Sample 2: input=[3,8,2,7]   target=[8,2,7,1]
      ...
    """
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    
    def __len__(self):
        # How many chunks can we make?
        return len(self.data) - self.block_size
    
    def __getitem__(self, idx):
        # Return one (input, target) pair
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

# Create datasets
block_size = 8
train_dataset = CharDataset(train_data, block_size)
val_dataset = CharDataset(val_data, block_size)

print(f"Training dataset:   {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")

# Look at one sample
x_sample, y_sample = train_dataset[0]
print(f"\nSample 0:")
print(f"  Input:  {x_sample.tolist()} → '{decode(x_sample.tolist())}'")
print(f"  Target: {y_sample.tolist()} → '{decode(y_sample.tolist())}'")

x_sample, y_sample = train_dataset[42]
print(f"\nSample 42:")
print(f"  Input:  {x_sample.tolist()} → '{decode(x_sample.tolist())}'")
print(f"  Target: {y_sample.tolist()} → '{decode(y_sample.tolist())}'")

print(f"\nThe Dataset just says: 'I have {len(train_dataset)} samples, ask me for any one by index.'")

### Now wrap it with a DataLoader — this handles batching and shuffling:

In [ ]:
# DataLoader: wraps a Dataset and gives you batches

train_loader = DataLoader(
    train_dataset,
    batch_size=4,       # 4 chunks per batch
    shuffle=True,        # randomize order each epoch (helps training)
)

# Iterate through one batch
for batch_idx, (xb, yb) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"  xb shape: {xb.shape}  (batch_size × block_size)")
    print(f"  yb shape: {yb.shape}")
    
    for i in range(len(xb)):
        print(f"  Chunk {i}: '{decode(xb[i].tolist())}' → '{decode(yb[i].tolist())}'")
    
    if batch_idx >= 2:  # just show first 3 batches
        print(f"\n  ... {len(train_loader)} total batches per epoch")
        break

print(f"\nThe DataLoader:")
print(f"  - Splits {len(train_dataset)} samples into batches of 4")
print(f"  - {len(train_loader)} batches per epoch")
print(f"  - Shuffles the order each epoch (different random batches!)")

## 6. Visualizing Tokens — What the Model Sees

Let's make it concrete. Here's the EXACT journey from text to model input:

In [ ]:
# The full pipeline visualized

sample_text = "To be or not to be"
print("THE FULL TEXT → NUMBERS PIPELINE")
print("=" * 60)

print(f"\nStep 1 — Raw text:")
print(f"  '{sample_text}'")

print(f"\nStep 2 — Split into characters (tokens):")
tokens = list(sample_text)
print(f"  {tokens}")

print(f"\nStep 3 — Map each to a number (token ID):")
ids = encode(sample_text)
for ch, idx in zip(tokens, ids):
    print(f"  '{ch}' → {idx}")

print(f"\nStep 4 — As a PyTorch tensor:")
tensor = torch.tensor(ids)
print(f"  {tensor}")
print(f"  shape: {tensor.shape}, dtype: {tensor.dtype}")

print(f"\nStep 5 — Split into input/target:")
x = tensor[:-1]
y = tensor[1:]
print(f"  Input:  {x.tolist()}  (all except last)")
print(f"  Target: {y.tolist()}  (all except first)")

print(f"\nStep 6 — Decode back (after model predicts):")
print(f"  {ids} → '{decode(ids)}'")

print(f"\n" + "=" * 60)
print(f"This is EXACTLY what happens inside ChatGPT,")
print(f"just with subword tokens instead of characters,")
print(f"and 50,000 vocabulary entries instead of {vocab_size}.")

## 7. Mini Project: Complete Data Pipeline for a Text File

Let's build a reusable pipeline that takes ANY text file and prepares it for training. This is the foundation we'll use when we build our actual model in later days.

In [ ]:
# Complete, reusable text data pipeline

class TextDataPipeline:
    """Everything you need to go from raw text to training-ready batches."""
    
    def __init__(self, text, block_size=8, batch_size=4, train_split=0.9):
        self.block_size = block_size
        self.batch_size = batch_size
        
        # Build vocabulary from the text
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}
        
        # Encode full text
        self.data = torch.tensor(self.encode(text), dtype=torch.long)
        
        # Train/val split
        split = int(train_split * len(self.data))
        self.train_data = self.data[:split]
        self.val_data = self.data[split:]
    
    def encode(self, text):
        return [self.char_to_idx[ch] for ch in text]
    
    def decode(self, ids):
        return ''.join([self.idx_to_char[i] for i in ids])
    
    def get_batch(self, split='train'):
        """Get a random batch of (input, target) pairs."""
        d = self.train_data if split == 'train' else self.val_data
        starts = torch.randint(0, len(d) - self.block_size, (self.batch_size,))
        x = torch.stack([d[s : s + self.block_size] for s in starts])
        y = torch.stack([d[s+1 : s + self.block_size + 1] for s in starts])
        return x, y
    
    def info(self):
        print(f"Text Data Pipeline")
        print(f"  Vocab size:     {self.vocab_size} unique characters")
        print(f"  Total tokens:   {len(self.data):,}")
        print(f"  Train tokens:   {len(self.train_data):,}")
        print(f"  Val tokens:     {len(self.val_data):,}")
        print(f"  Block size:     {self.block_size}")
        print(f"  Batch size:     {self.batch_size}")
        print(f"  Batch shape:    ({self.batch_size}, {self.block_size})")

# Use it!
pipeline = TextDataPipeline(text, block_size=16, batch_size=4)
pipeline.info()

In [ ]:
# Get a training batch and inspect it

xb, yb = pipeline.get_batch('train')

print(f"Training batch:")
print(f"  Input shape:  {xb.shape}")
print(f"  Target shape: {yb.shape}\n")

for i in range(len(xb)):
    inp = pipeline.decode(xb[i].tolist())
    tgt = pipeline.decode(yb[i].tolist())
    print(f"  Chunk {i}:")
    print(f"    Input:  '{inp}'")
    print(f"    Target: '{tgt}'")
    print()

## 8. How GPT's Tokenizer Differs (Preview)

We built a character-level tokenizer (~30 vocabulary entries). Real LLMs use **subword tokenization** (BPE), which is smarter:

```
Character-level:  "unhappy" → ['u','n','h','a','p','p','y']  (7 tokens)
GPT's tokenizer:  "unhappy" → ['un', 'happy']                (2 tokens)
```

Subword tokenization:
- Common words stay whole: "the" → ["the"] (1 token)
- Rare words get split: "cryptocurrency" → ["crypt", "ocur", "rency"] (3 tokens)
- Handles any word, even made-up ones

We'll build BPE from scratch on Day 21. For now, character-level gives us the same STRUCTURE (text → numbers → batches → model) — just with smaller vocabulary.

Let's peek at what a real tokenizer looks like:

In [ ]:
# Let's simulate what a subword tokenizer does (conceptually)

print("Character-level (what we built):")
print(f"  'To be or not to be' → {len('To be or not to be')} tokens")
print(f"  Each character = 1 token")
print(f"  Vocabulary: {vocab_size} entries\n")

print("Word-level (simple split by spaces):")
words = "To be or not to be".split()
print(f"  'To be or not to be' → {len(words)} tokens: {words}")
print(f"  Problem: 'unfriendliness' would need its own vocab entry\n")

print("Subword-level (what GPT uses):")
print(f"  'To be or not to be' → ~6 tokens: ['To', ' be', ' or', ' not', ' to', ' be']")
print(f"  'unfriendliness' → ~3 tokens: ['un', 'friend', 'liness']")
print(f"  Vocabulary: ~50,000 entries")
print(f"  Best of both worlds: compact AND handles any word")

print(f"\nAll three follow the same pipeline:")
print(f"  text → tokens → token_ids → batches → model")
print(f"  The only difference is HOW you split the text into tokens.")

## 9. Character Frequency — What Tokens Look Like in Our Data

In [ ]:
# Let's see which characters appear most often

from collections import Counter

counts = Counter(text)
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])

chars_display = []
freqs = []

for ch, count in sorted_counts[:20]:
    display = repr(ch) if ch in ['\n', ' ', '\t'] else f"'{ch}'"
    chars_display.append(display)
    freqs.append(count)
    
plt.figure(figsize=(12, 4))
plt.bar(range(len(chars_display)), freqs, color='steelblue')
plt.xticks(range(len(chars_display)), chars_display, rotation=45)
plt.ylabel('Count')
plt.title('Character Frequency in Our Text')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Space is the most common 'character' — makes sense!")
print("'e' and 't' are the most common letters — matches English frequency.")
print("\nIn a real tokenizer, common sequences get merged into single tokens.")
print("'th' appears often → BPE would merge it into one token.")

---

## Exercises

1. **Different text:** Replace the Shakespeare text with song lyrics, a Wikipedia article, or Python code. How does the vocabulary change? How many unique characters?

2. **Block size experiment:** Try block_size=4 vs block_size=32. How does the number of training examples change? What are the tradeoffs?

3. **Word-level tokenizer:** Build a tokenizer that splits on spaces instead of characters. What's the vocabulary size? What happens with punctuation?

4. **Unknown characters:** What happens if you try to encode a character that's not in the vocabulary (e.g., an emoji)? How would you handle this?

---

## Key Takeaways

| Concept | What It Does |
|---------|-------------|
| **Tokenization** | Splits text into pieces and maps each to a number |
| **Vocabulary** | The lookup table: token ↔ number |
| **Encoding** | text → list of numbers |
| **Decoding** | list of numbers → text |
| **Block size** | How many tokens the model sees at once (context window) |
| **Batch size** | How many chunks are processed in parallel |
| **Input/Target** | Input = tokens[0:n], Target = tokens[1:n+1] (shifted by 1) |
| **Train/Val split** | 90%/10% to detect memorization vs real learning |
| **Dataset** | "I have N samples, ask me for any one" |
| **DataLoader** | "I'll give you batches of samples, shuffled" |

### The full pipeline:

```
Raw text → Tokenize → Encode to IDs → Split train/val → Batch → Model
"hello"  → ['h','e','l','l','o'] → [7,4,11,11,14] → train/val → (4,8) batches → training
```

### How today connects to the journey:

```
Day 1: Matrix math                    (the computation)
Day 2: Gradients                       (how to learn)
Day 3: PyTorch autograd               (automate learning)
Day 4: Data pipeline                   (prepare text for the model) ← YOU ARE HERE
Day 5: Plotting & debugging            (monitor training)
Day 6+: Build the actual model!        (neurons, layers, attention...)
```

**Tomorrow:** We'll learn to visualize training — loss curves, debugging common problems, and matplotlib techniques for ML.